# CONDOR Set-Transformer Training for FlexDC Behavior Labels — v2

This notebook keeps the **CONDOR architecture and workflow** while repairing the training pipeline that caused the earlier run to save a degraded final epoch and miss the W2 feasible region.

Direct neural labels remain:

- `log(mean normalized tracking error + 1e-6)`
- `log(p90 normalized tracking error + 1e-3)`
- one QoS violation probability `Pj` for every real job type

Known FlexDC costs are reconstructed analytically. This version adds:

- physics-derived job and bid features;
- context-stratified grouped splits;
- context/feasibility/boundary-balanced training batches;
- `4×` QoS loss weight;
- warmup + cosine learning-rate decay;
- gradient clipping and early stopping;
- latest, best-loss, best-objective, best-feasibility, and final checkpoints;
- exact resume support after a Colab interruption;
- W&B metrics by workload/family so global accuracy cannot hide zero W2 recall.

## 0. Environment and repository controls — RUN FIRST

In [ ]:
from pathlib import Path
import os

RUN_ENV = "colab"  # "colab" or "local_pc"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent

COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
COMDER_BRANCH = "main"
FORCE_RECLONE = False

USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", or "disabled"
WANDB_PROJECT = "flexdc-unified-training"
WANDB_ENTITY = "amenon06-boston-university"

# Primary controlled experiment: train the repaired behavior-label model on
# the historical broad + focused-W2 data. Other profiles are retained for
# later ablations without changing the notebook code.
DATASET_PROFILE = "old_plus_w2dense"  # "old_plus_w2dense", "sweep_v2", "custom"

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)
print("DATASET_PROFILE:", DATASET_PROFILE)
print("W&B:", USE_WANDB, WANDB_MODE)

## 1. Install dependencies — COLAB ONLY

In [ ]:
if RUN_ENV == "colab":
    %pip install -q wandb pandas numpy scipy scikit-learn matplotlib tabulate openpyxl

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone or update CONDOR-FLEXDC — COLAB ONLY

In [ ]:
import subprocess

if RUN_ENV == "colab":
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    COMDER_ROOT = WORKSPACE / "comder-main"
    if FORCE_RECLONE and COMDER_ROOT.exists():
        subprocess.run(["rm", "-rf", str(COMDER_ROOT)], check=True)
    if not COMDER_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", COMDER_BRANCH, COMDER_REPO_URL, str(COMDER_ROOT)],
            check=True,
        )
    else:
        print("Repository already exists; leaving local files unchanged:", COMDER_ROOT)
else:
    COMDER_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/comder-main")

print("COMDER_ROOT:", COMDER_ROOT)

## 3. Optional Google Drive copy — RUN ONLY WHEN NEEDED

The historical data is intentionally not assumed to be in GitHub. Copy the two CSVs and the three v2 Python files into the paths below, or place them there manually before running the required-file check.

In [ ]:
USE_GOOGLE_DRIVE_DATA = False

# Edit only when USE_GOOGLE_DRIVE_DATA=True.
DRIVE_OLD_RESULTS = "/content/drive/MyDrive/path/to/traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv"
DRIVE_OLD_DIAGNOSTICS = "/content/drive/MyDrive/path/to/traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv"
DRIVE_MODEL_V2 = "/content/drive/MyDrive/path/to/data_center_model_flexdc_behavior_v2.py"
DRIVE_UTILS_V2 = "/content/drive/MyDrive/path/to/am_flexdc_behavior_training_utilities_v2.py"
DRIVE_TESTS_V2 = "/content/drive/MyDrive/path/to/test_flexdc_behavior_training_v2.py"

if RUN_ENV == "colab" and USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    am_root = COMDER_ROOT / "am_flexdc"
    train_dir = am_root / "train"
    old_dir = am_root / "data" / "pilots" / "traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective"
    train_dir.mkdir(parents=True, exist_ok=True)
    old_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(DRIVE_OLD_RESULTS, old_dir / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv")
    shutil.copy2(DRIVE_OLD_DIAGNOSTICS, old_dir / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv")
    shutil.copy2(DRIVE_MODEL_V2, train_dir / "data_center_model_flexdc_behavior_v2.py")
    shutil.copy2(DRIVE_UTILS_V2, train_dir / "am_flexdc_behavior_training_utilities_v2.py")
    shutil.copy2(DRIVE_TESTS_V2, train_dir / "test_flexdc_behavior_training_v2.py")
    print("Copied data and v2 training files from Drive.")
else:
    print("Drive copy skipped.")

## 4. Resolve dataset paths and check required files

In [ ]:
import sys

AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
MODELS_DIR = AM_FLEXDC_ROOT / "models" / "flexdc_behavior_v2"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "training_runs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

profiles = {
    "old_plus_w2dense": {
        "pilot_dir": AM_FLEXDC_ROOT / "data" / "pilots" / "traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective",
        "results_name": "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv",
        "diagnostics_name": "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv",
        "deduplicate": False,
        "tag": "old_plus_w2dense_behavior_v2",
    },
    "sweep_v2": {
        "pilot_dir": AM_FLEXDC_ROOT / "data" / "pilots" / "flexdc_sweep_v2_paper_objective",
        "results_name": "flexdc_sweep_combined_grid_search_results.csv",
        "diagnostics_name": "flexdc_sweep_combined_grid_search_diagnostics.csv",
        "deduplicate": True,
        "tag": "sweep_v2_behavior_v2",
    },
}

if DATASET_PROFILE == "custom":
    PILOT_DIR = Path("/content/path/to/custom/dataset")
    RESULTS_CSV = PILOT_DIR / "results.csv"
    DIAGNOSTICS_CSV = PILOT_DIR / "diagnostics.csv"
    DEDUPLICATE = False
    DATASET_TAG = "custom_behavior_v2"
else:
    profile = profiles[DATASET_PROFILE]
    PILOT_DIR = profile["pilot_dir"]
    RESULTS_CSV = PILOT_DIR / profile["results_name"]
    DIAGNOSTICS_CSV = PILOT_DIR / profile["diagnostics_name"]
    DEDUPLICATE = bool(profile["deduplicate"])
    DATASET_TAG = profile["tag"]

MODEL_PY = TRAIN_DIR / "data_center_model_flexdc_behavior_v2.py"
UTILS_PY = TRAIN_DIR / "am_flexdc_behavior_training_utilities_v2.py"
TEST_PY = TRAIN_DIR / "test_flexdc_behavior_training_v2.py"

required = [MODEL_PY, UTILS_PY, TEST_PY, RESULTS_CSV, DIAGNOSTICS_CSV]
missing = [path for path in required if not path.exists()]
if missing:
    print("Missing required files:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError("Copy the missing files before continuing.")

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

print("All required files found.")
print("RESULTS_CSV:", RESULTS_CSV)
print("DIAGNOSTICS_CSV:", DIAGNOSTICS_CSV)
print("DEDUPLICATE:", DEDUPLICATE)
print("DATASET_TAG:", DATASET_TAG)

## 5. Compile and run structural tests — REQUIRED

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "py_compile", MODEL_PY.name, UTILS_PY.name, TEST_PY.name],
    cwd=TRAIN_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, TEST_PY.name],
    cwd=TRAIN_DIR,
    check=True,
)
print("Compilation and structural tests passed.")

## 6. Import model and utilities

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

from data_center_model_flexdc_behavior_v2 import (
    DataCenterBehaviorModel,
    FlexDCBehaviorModelConfig,
)
from am_flexdc_behavior_training_utilities_v2 import (
    FlexDCBehaviorConstants,
    prepare_behavior_data,
    train_behavior_model,
    final_behavior_evaluation,
    evaluate_behavior_loader,
    load_behavior_model_checkpoint,
    context_metrics_table,
    sample_prediction_rows,
    choose_device,
)


def show_table(df, caption=None, precision=4):
    styled = (
        df.style.hide(axis="index")
        .format(precision=precision)
        .set_properties(**{"text-align": "left", "white-space": "normal", "font-size": "12px"})
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#0f172a"), ("color", "white"), ("font-weight", "bold"), ("text-align", "left")]},
            {"selector": "td", "props": [("border", "1px solid #cbd5e1"), ("padding", "6px")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "17px"), ("font-weight", "bold"), ("text-align", "left")]},
        ])
    )
    if caption:
        styled = styled.set_caption(caption)
    display(styled)

## 7. W&B login

In [ ]:
run = None
if USE_WANDB and WANDB_MODE != "disabled":
    import getpass
    import wandb

    os.environ["WANDB_MODE"] = WANDB_MODE
    os.environ.pop("WANDB_BASE_URL", None)
    if WANDB_MODE == "online":
        key = None
        try:
            from google.colab import userdata
            key = userdata.get("WANDB_API_KEY")
        except Exception:
            pass
        if key:
            wandb.login(key=key, relogin=True, verify=True)
        else:
            try:
                wandb.login(relogin=True, verify=True)
            except Exception:
                key = getpass.getpass("Paste W&B API key: ")
                wandb.login(key=key, relogin=True, verify=True)
        print("W&B login verified.")
    else:
        print("W&B mode:", WANDB_MODE)
else:
    print("W&B disabled.")

## 8. Training configuration — EDIT HERE

Defaults implement the planned repaired run. `RESUME_FROM` can point to the `*_latest.pt` file after a Colab interruption; optimizer state and epoch history will resume exactly.

In [ ]:
RUN_NAME = f"condor_set_transformer_{DATASET_TAG}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data and grouped split.
BATCH_SIZE = 2048  # reduce to 1024 only if the GPU runs out of memory
HELDOUT_FRACTION = 0.30
SPLIT_SEED = 0
NUM_WORKERS = 0

# Balanced training sampler. Validation remains unweighted/natural.
SAMPLER_MODE = "balanced"
SAMPLER_NATURAL_FRACTION = 0.50
SAMPLER_CONTEXT_FRACTION = 0.25
SAMPLER_PRIORITY_FRACTION = 0.25
SAMPLER_FEASIBLE_BOOST = 4.0
SAMPLER_TRACKING_BOUNDARY_BOOST = 2.0
SAMPLER_QOS_BOUNDARY_BOOST = 3.0

# Training operations.
MAX_EPOCHS = 250
BASE_LR = 3e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 30
EARLY_STOPPING_MIN_DELTA = 1e-5

# Direct-label losses.
MEAN_TRACKING_LOSS_WEIGHT = 1.0
P90_TRACKING_LOSS_WEIGHT = 1.0
QOS_LOSS_WEIGHT = 4.0
TRACKING_BOUNDARY_MULTIPLIER = 2.0
QOS_BOUNDARY_MULTIPLIER = 2.0

# Checkpoint selection. All roles are saved; this role is restored at the end
# of training, then the next section compares every saved role.
RESTORE_ROLE_AFTER_TRAINING = "best_loss"
SELECTED_CHECKPOINT_ROLE = "best_feasibility"  # may be changed after the comparison table

CHECKPOINT_DIR = MODELS_DIR / RUN_NAME
CHECKPOINT_PREFIX = RUN_NAME
RESUME_FROM = None
# Example after interruption:
# RESUME_FROM = CHECKPOINT_DIR / f"{CHECKPOINT_PREFIX}_latest.pt"

MODEL_CONFIG = FlexDCBehaviorModelConfig(
    dim_job_mix=13,
    dim_dc_features=12,
    st_dim_hidden=512,
    st_num_heads=4,
    global_projection_dim=128,
    linear_dim_hidden=512,
    qos_projection_dim=256,
    skip_connections=True,
    layer_norm=True,
    include_masked_mean_pool=True,
)
CONSTANTS = FlexDCBehaviorConstants()

print("Run:", RUN_NAME)
print("Device:", DEVICE)
print("Dataset:", RESULTS_CSV)
print("Resume from:", RESUME_FROM)
print("Model config:", MODEL_CONFIG.to_dict())

## 9. Prepare, split, standardize, and audit the dataset

In [ ]:
behavior_data = prepare_behavior_data(
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    batch_size=BATCH_SIZE,
    heldout_fraction=HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
    num_workers=NUM_WORKERS,
    deduplicate=DEDUPLICATE,
    constants=CONSTANTS,
    sampler_mode=SAMPLER_MODE,
    sampler_seed=SPLIT_SEED,
    natural_fraction=SAMPLER_NATURAL_FRACTION,
    context_fraction=SAMPLER_CONTEXT_FRACTION,
    priority_fraction=SAMPLER_PRIORITY_FRACTION,
    feasible_boost=SAMPLER_FEASIBLE_BOOST,
    tracking_boundary_boost=SAMPLER_TRACKING_BOUNDARY_BOOST,
    qos_boundary_boost=SAMPLER_QOS_BOUNDARY_BOOST,
)

summary = pd.DataFrame([
    {"Item": "Original rows", "Value": behavior_data.metadata.original_row_count},
    {"Item": "Rows after cleanup", "Value": behavior_data.metadata.deduplicated_row_count},
    {"Item": "Training rows", "Value": behavior_data.metadata.train_row_count},
    {"Item": "Heldout rows", "Value": behavior_data.metadata.heldout_row_count},
    {"Item": "Training groups", "Value": behavior_data.metadata.train_group_count},
    {"Item": "Heldout groups", "Value": behavior_data.metadata.heldout_group_count},
    {"Item": "Group overlap", "Value": behavior_data.audit["group_overlap"]},
    {"Item": "Training actual-feasible rows", "Value": behavior_data.audit["train_actual_feasible_rows"]},
    {"Item": "Heldout actual-feasible rows", "Value": behavior_data.audit["heldout_actual_feasible_rows"]},
])
show_table(summary, "Dataset and grouped-split audit", precision=4)

sampler_table = pd.DataFrame([
    {"Metric": key, "Value": value}
    for key, value in behavior_data.audit["sampler"].items()
    if isinstance(value, (int, float))
])
show_table(sampler_table, "Balanced-sampler audit", precision=4)

context_split = pd.DataFrame(behavior_data.audit["context_split_summary"])
show_table(context_split, "Per-context grouped split", precision=0)

print("Token features:", behavior_data.metadata.token_feature_names)
print("Global features:", behavior_data.metadata.global_feature_names)
print("Direct labels:", behavior_data.metadata.direct_label_names)

## 10. Create the repaired CONDOR behavior model

In [ ]:
model = DataCenterBehaviorModel(MODEL_CONFIG)
print(model)
print(f"Parameters: {model.parameter_count():,}")

# One real batch shape check before W&B/training.
real_batch = next(iter(behavior_data.train_loader))
with torch.no_grad():
    smoke_output = model(real_batch["features"], real_batch["workload"], real_batch["mask"])
print("Global input shape:", tuple(real_batch["features"].shape))
print("Token input shape:", tuple(real_batch["workload"].shape))
print("Tracking output shape:", tuple(smoke_output["tracking_logs"].shape))
print("QoS output shape:", tuple(smoke_output["qos_probabilities"].shape))

## 11. Start W&B run

In [ ]:
training_config = {
    "dataset_profile": DATASET_PROFILE,
    "dataset_tag": DATASET_TAG,
    "results_csv": str(RESULTS_CSV),
    "diagnostics_csv": str(DIAGNOSTICS_CSV),
    "batch_size": BATCH_SIZE,
    "heldout_fraction": HELDOUT_FRACTION,
    "split_seed": SPLIT_SEED,
    "sampler_mode": SAMPLER_MODE,
    "sampler_natural_fraction": SAMPLER_NATURAL_FRACTION,
    "sampler_context_fraction": SAMPLER_CONTEXT_FRACTION,
    "sampler_priority_fraction": SAMPLER_PRIORITY_FRACTION,
    "sampler_feasible_boost": SAMPLER_FEASIBLE_BOOST,
    "sampler_tracking_boundary_boost": SAMPLER_TRACKING_BOUNDARY_BOOST,
    "sampler_qos_boundary_boost": SAMPLER_QOS_BOUNDARY_BOOST,
    "max_epochs": MAX_EPOCHS,
    "base_lr": BASE_LR,
    "min_lr": MIN_LR,
    "warmup_epochs": WARMUP_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "mean_tracking_loss_weight": MEAN_TRACKING_LOSS_WEIGHT,
    "p90_tracking_loss_weight": P90_TRACKING_LOSS_WEIGHT,
    "qos_loss_weight": QOS_LOSS_WEIGHT,
    "tracking_boundary_multiplier": TRACKING_BOUNDARY_MULTIPLIER,
    "qos_boundary_multiplier": QOS_BOUNDARY_MULTIPLIER,
    "device": DEVICE,
    "model_config": MODEL_CONFIG.to_dict(),
    "direct_labels": behavior_data.metadata.direct_label_names,
    "token_features": behavior_data.metadata.token_feature_names,
    "global_features": behavior_data.metadata.global_feature_names,
}

if USE_WANDB and WANDB_MODE != "disabled":
    import wandb

    resume_wandb_id = None
    if RESUME_FROM and Path(RESUME_FROM).exists():
        resume_payload = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
        resume_wandb_id = resume_payload.get("wandb_run_id")

    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        mode=WANDB_MODE,
        config=training_config,
        id=resume_wandb_id,
        resume="must" if resume_wandb_id else None,
    )
    run.summary["data/train_rows"] = behavior_data.metadata.train_row_count
    run.summary["data/heldout_rows"] = behavior_data.metadata.heldout_row_count
    run.summary["data/group_overlap"] = behavior_data.audit["group_overlap"]
    print("W&B run:", run.url if hasattr(run, "url") else run.id)
else:
    run = None
    print("W&B run disabled.")

## 12. Train with scheduling, clipping, early stopping, and resumable checkpoints

A checkpoint is saved at every epoch before the next epoch starts. A browser disconnect does not lose the current run, and a terminated runtime can resume from `*_latest.pt`.

In [ ]:
model, training_result = train_behavior_model(
    model,
    behavior_data,
    epochs=MAX_EPOCHS,
    base_lr=BASE_LR,
    min_lr=MIN_LR,
    warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    device_name=DEVICE,
    mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
    qos_weight=QOS_LOSS_WEIGHT,
    tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
    qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_prefix=CHECKPOINT_PREFIX,
    model_config=MODEL_CONFIG.to_dict(),
    training_config=training_config,
    resume_from=RESUME_FROM,
    restore_role=RESTORE_ROLE_AFTER_TRAINING,
    wandb_run=run,
    metrics_every_n_epochs=1,
    verbose=True,
)

history = training_result.history
HISTORY_CSV = RESULTS_DIR / f"{RUN_NAME}_history.csv"
history.to_csv(HISTORY_CSV, index=False)

checkpoint_table = pd.DataFrame([
    {"Role": role, "Path": path, "Exists": Path(path).exists()}
    for role, path in training_result.checkpoint_paths.items()
])
show_table(checkpoint_table, "Saved checkpoints")
print("Training summary:")
print(json.dumps(training_result.summary, indent=2))
print("History:", HISTORY_CSV)

## 13. Compare every saved checkpoint on the same heldout split

In [ ]:
device = choose_device(DEVICE)
comparison_rows = []
checkpoint_metrics = {}

for role in ["best_loss", "best_objective", "best_feasibility", "final"]:
    path = Path(training_result.checkpoint_paths[role])
    if not path.exists():
        continue
    candidate, checkpoint = load_behavior_model_checkpoint(
        path,
        model_class=DataCenterBehaviorModel,
        config_class=FlexDCBehaviorModelConfig,
        device_name=DEVICE,
    )
    metrics, _ = evaluate_behavior_loader(
        candidate,
        behavior_data.heldout_loader,
        device=device,
        metadata=behavior_data.metadata,
        prefix="heldout",
        mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
        p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
        qos_weight=QOS_LOSS_WEIGHT,
        tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
        qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
        return_rows=False,
        include_workload_metrics=True,
    )
    checkpoint_metrics[role] = metrics
    comparison_rows.append({
        "Role": role,
        "Epoch": checkpoint.get("epoch"),
        "Heldout loss": metrics["heldout/loss/total"],
        "Objective R2": metrics["heldout/cost/full_objective/r2"],
        "Objective Spearman": metrics["heldout/cost/full_objective/spearman"],
        "P90 physical R2": metrics["heldout/tracking/p90_physical/r2"],
        "Per-job Pj MAE": metrics["heldout/qos/per_job_probability/mae"],
        "Max-Pj MAE": metrics["heldout/qos/max_probability/mae"],
        "Feasible precision": metrics["heldout/feasibility/combined/feasible_precision"],
        "Feasible recall": metrics["heldout/feasibility/combined/actual_feasible_accuracy"],
        "Feasible F1": metrics["heldout/feasibility/combined/f1_feasible"],
        "False-feasible rate": metrics["heldout/feasibility/combined/false_feasible_rate"],
    })

checkpoint_comparison = pd.DataFrame(comparison_rows)
show_table(checkpoint_comparison, "Heldout comparison of saved checkpoints", precision=5)
CHECKPOINT_COMPARISON_CSV = RESULTS_DIR / f"{RUN_NAME}_checkpoint_comparison.csv"
checkpoint_comparison.to_csv(CHECKPOINT_COMPARISON_CSV, index=False)

if run is not None:
    import wandb
    run.log({"checkpoint_comparison": wandb.Table(dataframe=checkpoint_comparison)})

print("Configured selected role:", SELECTED_CHECKPOINT_ROLE)

## 14. Load selected checkpoint and run final detailed evaluation

In [ ]:
selected_path = Path(training_result.checkpoint_paths.get(SELECTED_CHECKPOINT_ROLE, ""))
if not selected_path.exists():
    print(f"{SELECTED_CHECKPOINT_ROLE} is unavailable; falling back to best_loss.")
    SELECTED_CHECKPOINT_ROLE = "best_loss"
    selected_path = Path(training_result.checkpoint_paths[SELECTED_CHECKPOINT_ROLE])

model, selected_checkpoint = load_behavior_model_checkpoint(
    selected_path,
    model_class=DataCenterBehaviorModel,
    config_class=FlexDCBehaviorModelConfig,
    device_name=DEVICE,
)

metrics, train_predictions, heldout_predictions = final_behavior_evaluation(
    model,
    behavior_data,
    device_name=DEVICE,
    mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
    qos_weight=QOS_LOSS_WEIGHT,
    tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
    qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
)
context_summary = context_metrics_table(heldout_predictions)

METRICS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_metrics.csv"
TRAIN_PREDICTIONS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_train_predictions.csv"
HELDOUT_PREDICTIONS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_heldout_predictions.csv"
CONTEXT_SUMMARY_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_context_summary.csv"

pd.DataFrame([metrics]).to_csv(METRICS_CSV, index=False)
train_predictions.to_csv(TRAIN_PREDICTIONS_CSV, index=False)
heldout_predictions.to_csv(HELDOUT_PREDICTIONS_CSV, index=False)
context_summary.to_csv(CONTEXT_SUMMARY_CSV, index=False)

key_metrics = pd.DataFrame([
    {"Metric": "Selected role", "Value": SELECTED_CHECKPOINT_ROLE},
    {"Metric": "Selected epoch", "Value": selected_checkpoint.get("epoch")},
    {"Metric": "Heldout objective R2", "Value": metrics["heldout/cost/full_objective/r2"]},
    {"Metric": "Heldout objective Spearman", "Value": metrics["heldout/cost/full_objective/spearman"]},
    {"Metric": "Heldout p90 physical R2", "Value": metrics["heldout/tracking/p90_physical/r2"]},
    {"Metric": "Heldout per-job Pj MAE", "Value": metrics["heldout/qos/per_job_probability/mae"]},
    {"Metric": "Heldout max-Pj MAE", "Value": metrics["heldout/qos/max_probability/mae"]},
    {"Metric": "Feasible precision", "Value": metrics["heldout/feasibility/combined/feasible_precision"]},
    {"Metric": "Feasible recall", "Value": metrics["heldout/feasibility/combined/actual_feasible_accuracy"]},
    {"Metric": "Feasible F1", "Value": metrics["heldout/feasibility/combined/f1_feasible"]},
    {"Metric": "False-feasible rate", "Value": metrics["heldout/feasibility/combined/false_feasible_rate"]},
])
show_table(key_metrics, "Selected checkpoint — final heldout metrics", precision=5)
show_table(context_summary, "Heldout feasibility and error by workload/server/utilization context", precision=5)

print("Saved:")
for path in [METRICS_CSV, TRAIN_PREDICTIONS_CSV, HELDOUT_PREDICTIONS_CSV, CONTEXT_SUMMARY_CSV]:
    print(" -", path)

## 15. Send final metrics and tables to W&B

In [ ]:
if run is not None:
    import wandb

    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)) and np.isfinite(value):
            run.summary[f"selected/{key}"] = float(value)
    run.summary["selected_checkpoint_role"] = SELECTED_CHECKPOINT_ROLE
    run.summary["selected_checkpoint_epoch"] = int(selected_checkpoint.get("epoch", -1))
    run.summary["selected_checkpoint_path"] = str(selected_path)

    run.log({
        "heldout_context_summary": wandb.Table(dataframe=context_summary),
        "heldout_prediction_sample": wandb.Table(
            dataframe=sample_prediction_rows(heldout_predictions, max_rows=2048, seed=0)
        ),
    })
    print("W&B summaries and tables updated.")
else:
    print("W&B disabled.")

## 16. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

curves = [
    ("heldout/loss/total", "Heldout weighted loss"),
    ("learning_rate", "Learning rate"),
    ("heldout/cost/full_objective/r2", "Heldout objective R²"),
    ("heldout/feasibility/combined/feasible_precision", "Heldout feasible precision"),
    ("heldout/feasibility/combined/actual_feasible_accuracy", "Heldout feasible recall"),
    ("heldout/by_family/W2/feasibility/feasible_precision", "W2 feasible precision"),
    ("heldout/by_family/W2/feasibility/actual_feasible_accuracy", "W2 feasible recall"),
]
for column, title in curves:
    if column not in history.columns:
        continue
    plt.figure(figsize=(8, 4))
    plt.plot(history["epoch"], history[column])
    plt.xlabel("Epoch")
    plt.ylabel(title)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

## 17. Package artifacts and optionally download

In [ ]:
import zipfile

ARTIFACT_ZIP = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_artifacts.zip"
files_to_package = [
    HISTORY_CSV,
    CHECKPOINT_COMPARISON_CSV,
    METRICS_CSV,
    TRAIN_PREDICTIONS_CSV,
    HELDOUT_PREDICTIONS_CSV,
    CONTEXT_SUMMARY_CSV,
    selected_path,
]
# Also include all checkpoint roles so a later analysis can compare or resume.
for path in training_result.checkpoint_paths.values():
    p = Path(path)
    if p.exists() and p not in files_to_package:
        files_to_package.append(p)

with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_package:
        path = Path(path)
        if path.exists():
            archive.write(path, arcname=path.name)

print("Artifact ZIP:", ARTIFACT_ZIP)
print("Size MB:", ARTIFACT_ZIP.stat().st_size / 1024**2)

if run is not None:
    run.finish()

if RUN_ENV == "colab":
    try:
        from google.colab import files
        # Uncomment to download automatically:
        # files.download(str(ARTIFACT_ZIP))
        print("Use files.download(str(ARTIFACT_ZIP)) to download.")
    except Exception:
        pass